In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from dataclasses import asdict
from pathlib import Path
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import bsc_final_runtime

Config = bsc_final_runtime.Config
run_cell = bsc_final_runtime.run_cell
run_partA = bsc_final_runtime.run_partA


def default_run_dir() -> Path:
    return Path(os.environ.get("EXPERIMENT_BSC_OUTPUT_DIR", "outputs/experiment_bsc")).resolve()


def coverage_ci_lower_approx(R: int, nominal: float, alpha: float = 0.05) -> float:
    z = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    se = np.sqrt(nominal * (1.0 - nominal) / R)
    return float(nominal - z * se)


def save_fig(fig: plt.Figure, fig_dir: Path, stem: str) -> None:
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir / f"{stem}.png", dpi=160, bbox_inches="tight")
    fig.savefig(fig_dir / f"{stem}.pdf", bbox_inches="tight")


def plot_all(results: pd.DataFrame, diagnostics: pd.DataFrame, cfg: Config, fig_dir: Path) -> None:
    coverage_metric = "coverage_95"
    coverage_nominal = 0.95
    coverage_ci_lower = coverage_ci_lower_approx(cfg.R, coverage_nominal, alpha=0.05)

    coverage_min = min(float(results[coverage_metric].min()), coverage_ci_lower)
    coverage_max = max(float(results[coverage_metric].max()), coverage_nominal)
    if np.isclose(coverage_min, coverage_max):
        coverage_min -= 0.01
        coverage_max += 0.01

    n_values = sorted(int(v) for v in diagnostics["n"].unique())
    s_values = sorted(float(v) for v in diagnostics["S_true"].unique())

    fig, axes = plt.subplots(
        len(cfg.dgps),
        len(cfg.methods),
        figsize=(4.5 * len(cfg.methods), 3.3 * len(cfg.dgps)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for i, dgp in enumerate(cfg.dgps):
        for j, method in enumerate(cfg.methods):
            ax = axes[i, j]
            subset = results[(results["dgp"] == dgp) & (results["method"] == method)]
            for n in n_values:
                n_rows = subset[subset["n"] == int(n)].sort_values("S_true")
                n_rows = n_rows.dropna(subset=[coverage_metric])
                if n_rows.empty:
                    continue
                ax.plot(n_rows["S_true"], n_rows[coverage_metric], marker="o", label=f"n={n}")
            ax.axhline(coverage_nominal, linestyle=":", color="black", linewidth=1.0)
            ax.axhline(
                coverage_ci_lower,
                linestyle="-",
                color="red",
                linewidth=1.0,
                label="95% CI lower bound" if (i == 0 and j == 0) else None,
            )
            ax.grid(alpha=0.25, linestyle=":")
            ax.set_title(f"{dgp} | {method}")
            ax.set_xlabel("True Sharpe")
            ax.set_ylabel("95% coverage")
            ax.set_ylim(coverage_min, coverage_max)

    handles, labels = [], []
    for ax in axes.ravel():
        h, l = ax.get_legend_handles_labels()
        if h:
            handles, labels = h, l
            break
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 6))
    fig.tight_layout(rect=(0, 0, 1, 0.93 if handles else 1.0))
    save_fig(fig, fig_dir, "partA_coverage_95_by_n_grid")
    plt.show()
    plt.close(fig)

    reject_metric = "reject_rate_H0_S_eq_0"
    reject_nominal = cfg.alpha

    reject_min = min(float(results[reject_metric].min()), reject_nominal)
    reject_max = max(float(results[reject_metric].max()), reject_nominal)
    if np.isclose(reject_min, reject_max):
        reject_min -= 0.01
        reject_max += 0.01

    fig_r, axes_r = plt.subplots(
        len(cfg.dgps),
        len(cfg.methods),
        figsize=(4.5 * len(cfg.methods), 3.3 * len(cfg.dgps)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for i, dgp in enumerate(cfg.dgps):
        for j, method in enumerate(cfg.methods):
            ax = axes_r[i, j]
            subset = results[(results["dgp"] == dgp) & (results["method"] == method)]
            for n in n_values:
                n_rows = subset[subset["n"] == int(n)].sort_values("S_true")
                n_rows = n_rows.dropna(subset=[reject_metric])
                if n_rows.empty:
                    continue
                ax.plot(n_rows["S_true"], n_rows[reject_metric], marker="o", label=f"n={n}")
            ax.axhline(reject_nominal, linestyle=":", color="black", linewidth=1.0)
            ax.grid(alpha=0.25, linestyle=":")
            ax.set_title(f"{dgp} | {method}")
            ax.set_xlabel("True Sharpe")
            ax.set_ylabel("Rejection rate (H0: S = 0)")
            ax.set_ylim(reject_min, reject_max)

    handles_r, labels_r = [], []
    for ax in axes_r.ravel():
        h, l = ax.get_legend_handles_labels()
        if h:
            handles_r, labels_r = h, l
            break
    if handles_r:
        fig_r.legend(handles_r, labels_r, loc="upper center", ncol=min(len(labels_r), 6))
    fig_r.tight_layout(rect=(0, 0, 1, 0.93 if handles_r else 1.0))
    save_fig(fig_r, fig_dir, "partA_reject_rate_H0_S_eq_0_by_n_grid")
    plt.show()
    plt.close(fig_r)

    target_s_true = 0.5
    if s_values and not any(np.isclose(v, target_s_true) for v in s_values):
        target_s_true = min(s_values, key=lambda v: abs(v - target_s_true))

    fig2, axes2 = plt.subplots(
        len(cfg.dgps),
        1,
        figsize=(6.0, 3.3 * len(cfg.dgps)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for i, dgp in enumerate(cfg.dgps):
        ax = axes2[i, 0]
        subset = results[(results["dgp"] == dgp) & (np.isclose(results["S_true"], float(target_s_true)))]
        for method in cfg.methods:
            m_rows = subset[subset["method"] == method].sort_values("n")
            m_rows = m_rows.dropna(subset=[coverage_metric])
            if m_rows.empty:
                continue
            ax.plot(m_rows["n"], m_rows[coverage_metric], marker="o", label=method)
        ax.axhline(coverage_nominal, linestyle=":", color="black", linewidth=1.0)
        ax.axhline(
            coverage_ci_lower,
            linestyle="-",
            color="red",
            linewidth=1.0,
            label="95% CI lower bound" if i == 0 else None,
        )
        ax.grid(alpha=0.25, linestyle=":")
        ax.set_title(f"{dgp} | S_true={target_s_true:g}")
        ax.set_xlabel("n")
        ax.set_ylabel("95% coverage")
        ax.set_ylim(coverage_min, coverage_max)

    handles2, labels2 = [], []
    for ax in axes2.ravel():
        h, l = ax.get_legend_handles_labels()
        if h:
            handles2, labels2 = h, l
            break
    if handles2:
        fig2.legend(handles2, labels2, loc="upper center", ncol=min(len(labels2), 6))
    fig2.tight_layout(rect=(0, 0, 1, 0.93 if handles2 else 1.0))
    save_fig(fig2, fig_dir, f"partA_coverage_95_vs_n_s_true_{str(target_s_true).replace('.', 'p')}")
    plt.show()
    plt.close(fig2)

    for metric_col, metric_label in [("rmse", "RMSE"), ("bias", "Bias")]:
        metric_min = float(diagnostics[metric_col].min())
        metric_max = float(diagnostics[metric_col].max())
        if metric_col == "bias":
            metric_min = min(metric_min, 0.0)
            metric_max = max(metric_max, 0.0)
        if np.isclose(metric_min, metric_max):
            pad = 0.01 if np.isclose(metric_max, 0.0) else abs(metric_max) * 0.05
            metric_min -= pad
            metric_max += pad

        fig_s, axes_s = plt.subplots(
            len(cfg.dgps),
            1,
            figsize=(6.0, 3.3 * len(cfg.dgps)),
            sharex=True,
            sharey=True,
            squeeze=False,
        )
        for i, dgp in enumerate(cfg.dgps):
            ax = axes_s[i, 0]
            subset = diagnostics[diagnostics["dgp"] == dgp]
            for n in n_values:
                n_rows = subset[subset["n"] == int(n)].sort_values("S_true")
                n_rows = n_rows.dropna(subset=[metric_col])
                if n_rows.empty:
                    continue
                ax.plot(n_rows["S_true"], n_rows[metric_col], marker="o", label=f"n={n}")
            if metric_col == "bias":
                ax.axhline(0.0, linestyle=":", color="black", linewidth=1.0)
            ax.grid(alpha=0.25, linestyle=":")
            ax.set_title(f"{dgp}")
            ax.set_xlabel("True Sharpe")
            ax.set_ylabel(metric_label)
            ax.set_ylim(metric_min, metric_max)

        handles_s, labels_s = [], []
        for ax in axes_s.ravel():
            h, l = ax.get_legend_handles_labels()
            if h:
                handles_s, labels_s = h, l
                break
        if handles_s:
            fig_s.legend(handles_s, labels_s, loc="upper center", ncol=min(len(labels_s), 6))
        fig_s.tight_layout(rect=(0, 0, 1, 0.93 if handles_s else 1.0))
        save_fig(fig_s, fig_dir, f"partA_{metric_col}_vs_s_true_by_n_grid")
        plt.show()
        plt.close(fig_s)

        fig_n, axes_n = plt.subplots(
            len(cfg.dgps),
            1,
            figsize=(6.0, 3.3 * len(cfg.dgps)),
            sharex=True,
            sharey=True,
            squeeze=False,
        )
        for i, dgp in enumerate(cfg.dgps):
            ax = axes_n[i, 0]
            subset = diagnostics[diagnostics["dgp"] == dgp]
            for s_true in s_values:
                s_rows = subset[np.isclose(subset["S_true"], float(s_true))].sort_values("n")
                s_rows = s_rows.dropna(subset=[metric_col])
                if s_rows.empty:
                    continue
                ax.plot(s_rows["n"], s_rows[metric_col], marker="o", label=f"S_true={s_true:g}")
            if metric_col == "bias":
                ax.axhline(0.0, linestyle=":", color="black", linewidth=1.0)
            ax.grid(alpha=0.25, linestyle=":")
            ax.set_title(f"{dgp}")
            ax.set_xlabel("n")
            ax.set_ylabel(metric_label)
            ax.set_ylim(metric_min, metric_max)

        handles_n, labels_n = [], []
        for ax in axes_n.ravel():
            h, l = ax.get_legend_handles_labels()
            if h:
                handles_n, labels_n = h, l
                break
        if handles_n:
            fig_n.legend(handles_n, labels_n, loc="upper center", ncol=min(len(labels_n), 6))
        fig_n.tight_layout(rect=(0, 0, 1, 0.93 if handles_n else 1.0))
        save_fig(fig_n, fig_dir, f"partA_{metric_col}_vs_n_by_s_true_grid")
        plt.show()
        plt.close(fig_n)


def try_write_parquet(df: pd.DataFrame, path: Path) -> None:
    try:
        df.to_parquet(path, index=False)
    except Exception:
        pass


def env_versions() -> dict:
    try:
        import importlib.metadata as ilmd
    except Exception:
        ilmd = None

    pkgs = ["numpy", "pandas", "matplotlib", "pyarrow"]
    versions = {}
    for pkg in pkgs:
        try:
            versions[pkg] = ilmd.version(pkg) if ilmd else None
        except Exception:
            versions[pkg] = None
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }


def execute_partA(cfg: Config, run_dir: Path | str | None = None):
    run_dir = Path(run_dir) if run_dir is not None else default_run_dir()
    run_dir = run_dir.resolve()
    cache_dir = run_dir / "cache"
    fig_dir = run_dir / "figures"
    for path in (run_dir, cache_dir, fig_dir):
        path.mkdir(parents=True, exist_ok=True)

    cfg_payload = json.dumps(asdict(cfg), sort_keys=True, separators=(",", ":"))
    cfg_hash = hashlib.sha256(cfg_payload.encode("utf-8")).hexdigest()[:12]
    cache_csv = cache_dir / f"partA_results_{cfg_hash}.csv"
    cache_parquet = cache_dir / f"partA_results_{cfg_hash}.parquet"
    cache_diag_csv = cache_dir / f"partA_diagnostics_{cfg_hash}.csv"
    cache_diag_parquet = cache_dir / f"partA_diagnostics_{cfg_hash}.parquet"

    if cache_csv.exists() and cache_diag_csv.exists():
        results = pd.read_csv(cache_csv)
        diagnostics = pd.read_csv(cache_diag_csv)
        print(f"[cache] loaded {cache_csv}")
        print(f"[cache] loaded {cache_diag_csv}")
    else:
        results, diagnostics = run_partA(cfg)
        results.to_csv(cache_csv, index=False)
        diagnostics.to_csv(cache_diag_csv, index=False)
        try_write_parquet(results, cache_parquet)
        try_write_parquet(diagnostics, cache_diag_parquet)
        (run_dir / f"config_partA_{cfg_hash}.json").write_text(
            json.dumps(asdict(cfg), indent=2),
            encoding="utf-8",
        )
        (run_dir / f"env_versions_{cfg_hash}.json").write_text(
            json.dumps(env_versions(), indent=2),
            encoding="utf-8",
        )
        print(f"[run] wrote {cache_csv}")
        print(f"[run] wrote {cache_diag_csv}")

    legacy_reject_cols = [
        col
        for col in results.columns
        if col.startswith("reject_rate_H0_") and col.endswith("_0") and col != "reject_rate_H0_S_eq_0"
    ]
    if legacy_reject_cols and "reject_rate_H0_S_eq_0" not in results.columns:
        results = results.rename(columns={legacy_reject_cols[0]: "reject_rate_H0_S_eq_0"})

    for col in ["bias", "rmse", "mc_sd_S_hat"]:
        if col in results.columns:
            results = results.drop(columns=[col])

    latest_results = run_dir / "results_partA_cell_parametric_bootstrap.csv"
    latest_diagnostics = run_dir / "results_partA_diagnostics.csv"
    results.to_csv(latest_results, index=False)
    diagnostics.to_csv(latest_diagnostics, index=False)

    plot_all(results, diagnostics, cfg, fig_dir)

    artifact_paths = {
        "run_dir": run_dir,
        "cache_dir": cache_dir,
        "fig_dir": fig_dir,
        "results_csv": latest_results,
        "diagnostics_csv": latest_diagnostics,
    }
    return results, diagnostics, artifact_paths


In [ ]:
# Editable notebook config
MAX_WORKERS_DEFAULT = max(1, (os.cpu_count() or 2) - 1)

cfg = Config(
    seed=0,
    R=300,
    B_cell=300,
    dgps=("iid_normal", "garch11_t"),
    methods=("iid_normal_analytic", "cell_parametric_bootstrap_wald"),
    n_grid=(36, 60, 120),
    S_grid=(0.2, 0.4, 0.6, 0.8, 1.0),
    g_alpha=0.05,
    g_beta=0.90,
    nu=7.0,
    burn=500,
    max_workers=MAX_WORKERS_DEFAULT,
)

run_dir = default_run_dir()

print(f"run_dir = {run_dir}")
cfg


In [ ]:
results, diagnostics, artifact_paths = execute_partA(cfg, run_dir=run_dir)

print(results)
print(diagnostics)
print()
for name, path in artifact_paths.items():
    print(f"{name}: {path}")


In [ ]:
from time import perf_counter

# Step 1: choose trial values
R0 = 500
B0 = 300

trial_cfg = Config(**{**asdict(cfg), "R": R0, "B_cell": B0, "max_workers": 1})
trial_dgp = "garch11_t" if "garch11_t" in trial_cfg.dgps else trial_cfg.dgps[0]
trial_n = trial_cfg.n_grid[len(trial_cfg.n_grid) // 2]
trial_s_true = trial_cfg.S_grid[len(trial_cfg.S_grid) // 2]

# Step 2: time one representative cell
t0 = perf_counter()
trial_rows, trial_diag = run_cell(trial_dgp, trial_n, trial_s_true, asdict(trial_cfg))
elapsed_cell = perf_counter() - t0
t_rep_hat = elapsed_cell / R0

summary = pd.Series(
    {
        "R0": R0,
        "B0": B0,
        "representative_dgp": trial_dgp,
        "representative_n": trial_n,
        "representative_S_true": trial_s_true,
        "elapsed_cell_seconds": elapsed_cell,
        "t_rep_hat_seconds": t_rep_hat,
    },
    name="timing_trial",
)
summary
